# Sprint 0 - Data Acquisition & Exploration

**Project:** Smart Crop Disease Detection and Assistant (see `AGENT.md` / `final_brief_and_plan.md`)

**Goal of this notebook (Sprint 0 "done when"):** download both datasets, organize them into
`train/val/test` folders by class, and load a labeled batch of images.

| Dataset | Source | Images | Classes |
|---|---|---|---|
| PlantVillage | GitHub `spMohanty/PlantVillage-Dataset` (raw/color) | 54,305 | 38 |
| PlantDoc | GitHub `pratikkayal/PlantDoc-Dataset` | 2,578 | 28 |

**Storage design (important):** Google Drive gets **one archive file per dataset** (`plantvillage_raw.zip`,
`plantdoc_raw.zip`) plus a small sha256 manifest. Creating ~54k individual image files on Drive directly
exceeds Google's per-day file-request quota (the "Drive quota exceeded" error - it is NOT free space), and
Colab's FUSE cache can hide partial writes. So downloads happen on **local disk** and Drive only ever holds
the two archives; each session unzips them locally.

**How to use:** `Runtime -> Run all`.

**Pipeline:**
1. Mount Drive + clone this repo
2. Install dependencies
3. *(once)* Reset `folium/data` on Drive
4. `scripts/download_datasets.py` -> local raw in `/content/folium_raw`, uploads the 2 archives to Drive
5. Hydrate local raw from the Drive archives, then `scripts/organize_datasets.py` -> local `/content/folium_data`
6. Load a batch of images with `torchvision.ImageFolder` + `DataLoader` and inspect it
7. **Durability gate** - verify Drive's archives from a *fresh* session (see Step 7)

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip() or gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Google Drive and clone the repo

- **Drive** holds the two dataset archives (`folium/data`) so the data persists across sessions.
- **Repo** is cloned fresh (or reused) so the latest scripts under `scripts/` are available.
- **Local** (`/content/folium_raw`, `/content/folium_data`) is where raw + splits live this session.

In [ ]:
from google.colab import drive

# Mount Google Drive so the dataset archives persist across sessions.
# A popup asks you to authorize - click through and allow access.
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")  # durable archives only (2 zips + manifests)
LOCAL_RAW_DIR = Path("/content/folium_raw")           # per-session raw (unzipped from Drive)
LOCAL_DATA_DIR = Path("/content/folium_data")          # per-session organized splits

if not (REPO_DIR / "scripts").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    print(f"repo already cloned at {REPO_DIR}")

DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_RAW_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (Drive archives):", DATA_DIR)
print("LOCAL_RAW_DIR:", LOCAL_RAW_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)

## Step 2 - Install dependencies

Training deps (`torch`, `torchvision`, `albumentations`) are installed now so we
don't have to wait again in Sprint 1.

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless

## Step 3 - (Once) Reset the Drive data folder

On 11 Aug the first attempt left Drive with a partial dataset (FUSE cache hid the missing tail), and a
second attempt hit Google's per-day file-request quota because ~54k tiny files were written to Drive one
by one. The archives below fix both.

Set `RESET_DATA = True` **one time** to delete `folium/data` (only the 2 archives + manifests live there),
then flip it back to `False`. Never re-run this after the data is verified durable.

In [ ]:
RESET_DATA = True   # set True exactly once to wipe folium/data, then back to False

if RESET_DATA:
    !rm -rf {DATA_DIR}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Reset: deleted and recreated {DATA_DIR}")
else:
    print("RESET_DATA=False; keeping existing Drive data")

## Step 4 - Download datasets (local) and upload archives to Drive

Runs `scripts/download_datasets.py`:
- **PlantDoc first** (small, fast), cloned in `--work-dir`, moved to local `--data-dir`
- **PlantVillage**: sparse checkout of `raw/color` (~1 GB) into local `--data-dir`
- Uploads **one zip per dataset** to Drive (`--upload-dir`) + a sha256 manifest - 2 file writes total
- **Resumable**: only classes whose local count differs from expected are re-fetched
- Skips entirely when the Drive archive is already verified (fresh sessions go straight to Step 5)

> If Google still reports the upload quota as exceeded right now, the download still completes locally
> and organizes fine; just re-run `Step 4` (or the upload via `--upload-dir`) once the quota resets.

In [ ]:
import subprocess
import sys

WORK_DIR = Path("/content/folium_cache")  # session-only scratch (clones/checkouts)
WORK_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "download_datasets.py"),
    "--data-dir", str(LOCAL_RAW_DIR),
    "--work-dir", str(WORK_DIR),
    "--upload-dir", str(DATA_DIR),
    "--dataset", "all",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "download_datasets.py failed"

## Step 5 - Hydrate local raw, then organize into train/val/test (local)

1. `hydrate_dataset()` unzips the Drive archives into `/content/folium_raw` (skips if already present
   with the right counts - e.g. right after Step 4).
2. `organize_datasets.py` reads raw from local and writes the splits **locally**:
   - **PlantVillage**: stratified 80/10/10 split (seed 42)
   - **PlantDoc**: keeps its shipped test set, carves 10% of training images as validation
   - Writes `class_map.json` (PlantDoc -> PlantVillage class mapping)

Local splits avoid any Drive writes per session; they are rebuilt deterministically from the archives.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))

from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"

## Step 6 - Verify: load a labeled batch

The "done" check for Sprint 0: load images + labels with `torchvision.ImageFolder`
and pull one batch from a `DataLoader`.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
pv_train = datasets.ImageFolder(LOCAL_DATA_DIR / "plantvillage" / "train", transform=transform)
loader = DataLoader(pv_train, batch_size=32, shuffle=True, num_workers=2)

images, labels = next(iter(loader))
print("One batch: images", tuple(images.shape), "labels", tuple(labels.shape))
print("dtype/range: images", images.dtype, round(float(images.min()), 2), "-", round(float(images.max()), 2))
print("num train classes:", len(pv_train.classes))
print("num train images:", len(pv_train))
assert len(pv_train) > 0 and len(pv_train.classes) == 38
print("SPRINT 0 DONE: can load a labeled batch of images")

## Step 7 - Durability gate: verify Drive's archives

A download is only trustworthy once a **fresh** Colab session (no FUSE cache from the session that
wrote it) confirms the archives on Drive match their sha256 manifests and expected counts.

Run this cell now, then **open a brand-new session**, run Step 1 + this cell again, and confirm it
still says PASS there too.

> The expected tables come from `scripts/download_datasets.py`, measured from the upstream git trees
> with `git ls-tree` (so PlantDoc's counts include the 6 case-duplicate files that only a Linux/Colab
> clone keeps).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(REPO_DIR))

from scripts.download_datasets import (
    PLANTVILLAGE_EXPECTED,
    PLANTDOC_EXPECTED,
    _zip_verified,
    _class_counts,
    _plantdoc_counts,
)


def report_upload(name: str, expected: dict) -> list:
    manifest = _zip_verified(DATA_DIR, name)
    if manifest is None:
        return [f"no verified {name}_raw.zip on Drive (missing or sha256 mismatch)"]
    problems = []
    if manifest.get("classes") != expected:
        problems.append(f"{name} manifest classes != expected table")
    print(f"  {name}_raw.zip: {manifest.get('zip_bytes', 0) / 1e6:.0f} MB, sha256 {manifest.get('zip_sha256', '?')[:12]}...")
    return problems


print("Drive archives (authoritative durability check):")
problems = []
problems += report_upload("plantvillage", PLANTVILLAGE_EXPECTED)
problems += report_upload("plantdoc", PLANTDOC_EXPECTED)

print("\nLocal hydration cross-check (informational; raw is hydrated in Step 5):")
pv_raw = LOCAL_RAW_DIR / "plantvillage" / "raw"
pd_raw = LOCAL_RAW_DIR / "plantdoc" / "raw"
print("  plantvillage/raw:", "OK" if _class_counts(pv_raw) == PLANTVILLAGE_EXPECTED else ("not hydrated yet" if not pv_raw.exists() else "MISMATCH"))
print("  plantdoc/raw:", "OK" if _plantdoc_counts(pd_raw) == PLANTDOC_EXPECTED else ("not hydrated yet" if not pd_raw.exists() else "MISMATCH"))

if not problems:
    print("\nDURABILITY GATE: PASS in this session")
    print("NOW open a NEW Colab session and re-run Step 1 + this cell.")
    print("If it PASSES there too, the archives are durably on Drive.")
    print("If it FAILS there, re-run Step 4 (download --upload-dir) in that session.")
else:
    print("\nDURABILITY GATE: FAIL")
    for p in problems:
        print("  -", p)
    print("Re-run Step 4 (download --upload-dir) before trusting the data.")

### Sample images from the batch (resized to 256x256)

In [ ]:
import matplotlib.pyplot as plt
import torchvision.utils as vutils

grid = vutils.make_grid(images[:16], nrow=4, normalize=True).permute(1, 2, 0).numpy()
plt.figure(figsize=(12, 12))
plt.imshow(grid)
plt.axis("off")
plt.title("First 16 images (resized 256x256)")
plt.show()

for i in range(8):
    print(f"  {i}: {pv_train.classes[labels[i].item()]}")

### Train class distribution (PlantVillage)

In [ ]:
import collections

counts = collections.Counter(pv_train.targets)
ordered = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
names = [pv_train.classes[i] for i, _ in ordered]
values = [v for _, v in ordered]

plt.figure(figsize=(16, 5))
plt.bar(names, values)
plt.xticks(rotation=90, fontsize=8)
plt.ylabel("images")
plt.title("PlantVillage train class distribution (n = %d)" % len(pv_train))
plt.show()

## Where things live

**Durable on Google Drive** (2 archives + manifests - what the Step 7 gate verifies by sha256):
```
<DATA_DIR> = /content/drive/MyDrive/folium/data
  plantvillage_raw.zip        54,305 images / 38 classes (~1 GB)
  plantvillage_raw.manifest.json
  plantdoc_raw.zip            2,578 images / 28 train + 27 test classes
  plantdoc_raw.manifest.json
```

**Per-session, local** (rebuilt each run - deterministic, no Drive writes):
```
<LOCAL_RAW_DIR> = /content/folium_raw      raw, unzipped from the Drive archives
<LOCAL_DATA_DIR> = /content/folium_data    organized splits + class_map.json
  plantvillage/{train,val,test}/<class>/*.jpg   80/10/10, seed 42
  plantdoc/{train,val,test}/<class>/*.jpg       shipped test + 90/10
```

**Notes**
- `WORK_DIR` (`/content/folium_cache`) is session-only scratch - wiped on reset, as expected.
- The Step 7 durability gate is what tells you the archives actually committed to Drive - the
  session's own reads can lie (FUSE cache), so always re-check from a fresh session.
- Google's Drive quota that was hit on 11 Aug is the **per-day file-request** quota, not free space.
  Storing two archives instead of ~54k files avoids it entirely.